# Prime Factorization via Quantum Energy Landscape

Factors **N = 35** by minimising the continuous energy landscape $(p \cdot q - N)^2$ combined
with integer-locking harmonics $-\cos(2\pi p) - \cos(2\pi q)$ that create local minima at
every integer pair. Recovers $p = 5$, $q = 7$.

This is a **quantum-inspired optimization** technique — the same energy-landscape formulation
used by D-Wave quantum annealers to embed integer factoring as a QUBO problem.

**No GPU required. Runs in under 100 ms.**

In [ ]:
# ── Set your API key ───────────────────────────────────────────────────────────
# Get a free key: curl -s -X POST https://api.qumulator.com/keys \
#   -H 'Content-Type: application/json' -d '{"name": "my-key"}' | python -m json.tool

import os
API_KEY = os.environ.get('QUMULATOR_API_KEY', 'YOUR_KEY_HERE')
API_URL = 'https://api.qumulator.com'

In [ ]:
%pip install qumulator-sdk scipy --quiet
from qumulator import QumulatorClient
import numpy as np
from scipy.optimize import minimize
import time

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print('SDK ready.')

## Energy Landscape

The cost function creates a 2D landscape with deep wells at integer $(p, q)$ pairs where
$p \times q = N$:

$$E(p, q) = (p \cdot q - N)^2 - \lambda\bigl[\cos(2\pi p) + \cos(2\pi q)\bigr]$$

The harmonic terms lock the optimizer to integers, so L-BFGS-B finds the exact factorization.

In [ ]:
N_target = 35

def energy(x, lam=0.5):
    p, q = x
    main   = (p * q - N_target) ** 2
    lock_p = -np.cos(2 * np.pi * p)
    lock_q = -np.cos(2 * np.pi * q)
    return main + lam * (lock_p + lock_q)

# Grid search for best starting point, then L-BFGS-B refinement
best_E, best_res = np.inf, None
t0 = time.perf_counter()

for p0 in range(2, 10):
    for q0 in range(p0, N_target // p0 + 1):
        res = minimize(energy, [p0 + 0.5, q0 + 0.5],
                       method='L-BFGS-B',
                       bounds=[(2, N_target), (2, N_target)],
                       options={'ftol': 1e-12, 'maxiter': 1000})
        if res.fun < best_E:
            best_E, best_res = res.fun, res

t_total = time.perf_counter() - t0
p_found, q_found = best_res.x
p_int, q_int = int(round(p_found)), int(round(q_found))

print(f'Continuous converged : p = {p_found:.6f}, q = {q_found:.6f}')
print(f'Rounded to integers  : p = {p_int}, q = {q_int}')
print(f'Check: {p_int} × {q_int} = {p_int * q_int}')
print(f'Time : {t_total*1000:.1f} ms')

In [ ]:
print('\n' + '='*50)
print(' BENCHMARK: Prime Factorization via Energy Landscape')
print('='*50)
correct = (p_int * q_int == N_target)
print(f'  Target : {N_target}  →  Found: {p_int} × {q_int} = {p_int * q_int}')
print(f'  Result : {"✓ PASS — factored correctly" if correct else "✗ FAIL"}')
print(f'  Time   : {t_total*1000:.1f} ms')
assert correct, f'Factorization failed: {p_int} × {q_int} ≠ {N_target}'